<a href="https://colab.research.google.com/github/jakemaier1/GA_Capstone_2026/blob/main/GA_Taxiway_robot_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

!pip install -q ultralytics

Mon Nov  9 21:22:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import zipfile
import os

# Updated to match the actual file present in your Colab files
zip_file_name = 'Airport Sign Labels.v2-five-signs-180-images-yolov8.yolov8.zip'
extract_path = '/content/runway_dataset'

if os.path.exists(zip_file_name):
    with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✅ Dataset successfully extracted to local SSD:", extract_path)
else:
    print(f"❌ Could not find '{zip_file_name}'. Make sure you uploaded it to the Colab file sidebar and that it includes '.zip' at the end!")

✅ Dataset successfully extracted to local SSD: /content/runway_dataset


In [ ]:
import yaml

yaml_path = '/content/runway_dataset/data.yaml'

# Load Roboflow's yaml
with open(yaml_path, 'r') as f:
    config = yaml.safe_load(f)

# Update root path for Colab local storage
config['path'] = '/content/runway_dataset'

# Save it back
with open(yaml_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("✅ data.yaml configured successfully!")
print(f"Detected Sign Classes: {config.get('names')}")

✅ data.yaml configured successfully!
Detected Sign Classes: ['Runway_36_Direction_Right', 'Runway_36_Red', 'Taxiway_B', 'Taxiway_B_Direction_Left', 'Terminal_A_Direction_Right']


In [ ]:
from ultralytics import YOLO

# Load pre-trained nano backbone
model = YOLO('yolov8n.pt')

# Train on your custom sign dataset
results = model.train(
    data='/content/runway_dataset/data.yaml',
    epochs=30,             # 30 epochs is ideal for a small initial test dataset
    imgsz=416,             # 416x416 optimized for Jetson Nano speed
    batch=8,               # Small batch size for efficient steps
    name='runway_signs_v1',
    plots=True
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/runway_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0

In [ ]:
# Evaluate model performance on the validation split
metrics = model.val()

print("\n" + "="*40)
print(f"mAP50-95 (Overall Accuracy): {metrics.box.map:.4f}")
print(f"mAP50 (Accuracy at 50% IoU): {metrics.box.map50:.4f}")
print("="*40)

# Download best.pt directly to your computer
from google.colab import files
weights_path = 'runs/detect/runway_signs_v1/weights/best.pt'

if os.path.exists(weights_path):
    files.download(weights_path)
    print("📥 Downloading best.pt to your computer...")
else:
    print(f"❌ Weights not found at {weights_path}. Check if training completed successfully.")

Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,133,679 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 960.4±320.0 MB/s, size: 26.2 KB)
val: Scanning /content/runway_dataset/valid/labels.cache... 36 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 36/36 10.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.2it/s 1.4s
                   all         36         36      0.927      0.931      0.984      0.893
Runway_36_Direction_Right          8          8      0.807          1      0.995      0.908
         Runway_36_Red          6          6      0.964          1      0.995      0.896
             Taxiway_B         11         11      0.968          1      0.995      0.947
Taxiway_B_Direction_Left          6          6          1      0.657      0.942      0.846
Terminal_A_Direction_Right          5     

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Downloading best.pt to your computer...


In [ ]:
from ultralytics import YOLO
import os

# Load pre-trained YOLO11 nano model
model_yolo11 = YOLO('yolo11n.pt')

# Train on your custom airport sign dataset
results_yolo11 = model_yolo11.train(
    data='/content/runway_dataset/data.yaml',
    epochs=30,             # Keep the same training duration for direct comparison
    imgsz=416,             # Optimizing for Jetson Nano inference speed
    batch=8,
    name='runway_signs_yolo11_nano',
    plots=True
)

Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/runway_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=runway_signs_yolo11_nano, nbs=64, nms=None, opset=None, 

In [ ]:
# Evaluate performance of YOLO11 nano
metrics_yolo11 = model_yolo11.val()

print("\n" + "="*40)
print(f"YOLO11 Nano mAP50: {metrics_yolo11.box.map50:.4f}")
print(f"YOLO11 Nano mAP50-95: {metrics_yolo11.box.map:.4f}")
print("="*40)

# Download YOLO11 weights directly
from google.colab import files
yolo11_weights_path = 'runs/detect/runway_signs_yolo11_nano/weights/best.pt'

if os.path.exists(yolo11_weights_path):
    files.download(yolo11_weights_path)
    print("📥 Downloading your new YOLO11 best.pt weights to your computer...")
else:
    print("❌ YOLO11 weights not found. Verify training completed successfully.")

Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,598,023 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 837.3±210.1 MB/s, size: 26.2 KB)
val: Scanning /content/runway_dataset/valid/labels.cache... 36 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 36/36 6.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 3.6it/s 0.8s
                   all         36         36      0.972       0.94       0.99      0.907
Runway_36_Direction_Right          8          8      0.912          1      0.995      0.901
         Runway_36_Red          6          6      0.966          1      0.995      0.895
             Taxiway_B         11         11          1      0.953      0.995      0.951
Taxiway_B_Direction_Left          6          6          1      0.749      0.972      0.907
Terminal_A_Direction_Right          5   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Downloading your new YOLO11 best.pt weights to your computer...


### **Quick 10-Epoch Test Run**

Let's train a new YOLO11 Nano model for only **10 epochs** to see what accuracy we can achieve in a third of the time.

In [ ]:
from ultralytics import YOLO
import os

# Initialize pre-trained YOLO11 Nano
model_10epochs = YOLO('yolo11n.pt')

# Train for exactly 10 epochs
results_10epochs = model_10epochs.train(
    data='/content/runway_dataset/data.yaml',
    epochs=10,
    imgsz=416,
    batch=8,
    name='runway_signs_yolo11_10epochs',
    plots=True
)

Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/runway_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=runway_signs_yolo11_10epochs, nbs=64, nms=None, opset=No

In [ ]:
# Evaluate the 10-epoch model performance
metrics_10e = model_10epochs.val()

print("\n" + "="*40)
print(f"10-Epoch mAP50 (Accuracy): {metrics_10e.box.map50:.4f}")
print(f"10-Epoch mAP50-95 (Strict Accuracy): {metrics_10e.box.map:.4f}")
print("="*40)

# Download the 10-epoch weights to compare
from google.colab import files
ten_epoch_weights = 'runs/detect/runway_signs_yolo11_10epochs/weights/best.pt'

if os.path.exists(ten_epoch_weights):
    os.rename(ten_epoch_weights, 'runs/detect/runway_signs_yolo11_10epochs/weights/best_10_epochs.pt')
    files.download('runs/detect/runway_signs_yolo11_10epochs/weights/best_10_epochs.pt')
    print("📥 Downloaded best_10_epochs.pt!")
else:
    print("❌ Weights file not found.")